# Pre-label the Egyptian images (Colab)

Trains a labelling model on CHV, then predicts boxes for the 173 Egyptian images (decision D-004).
**These are pre-labels.** A person must review every image in CVAT before the labels are used.

How to run: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Takes about 40–60 minutes.
At the end, a file `egy_prelabels.zip` downloads to your computer (a few hundred KB).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics gdown

## 1. Settings

In [ ]:
SEED = 42
REPO_RAW = "https://raw.githubusercontent.com/YOUANNA13/-1_proj/main"
CHV_DRIVE_ID = "1fdGn67W0B7ShpBDbbQpUF0ScPQa4DR0a"   # official CHV link (Wang et al. repo)
EGY_DRIVE_ID = "1Y30bp2I8c5RGSxtRb0Xa6t5RjGmzxx8n"   # Egyptian images link (paper repo)

MODEL = "yolov8m.pt"   # strongest model that trains quickly on a T4
EPOCHS = 50
IMGSZ = 640
BATCH = 16
CONF = 0.25            # minimum confidence for a pre-label box

NAMES = ["person", "vest", "blue_helmet", "red_helmet", "white_helmet", "yellow_helmet"]  # D-005, never reorder

## 2. Download the data and our split lists

In [ ]:
import os, zipfile, subprocess, urllib.request, urllib.parse
from pathlib import Path

W = Path("/content/work"); W.mkdir(exist_ok=True)
os.chdir(W)

def gdrive(file_id, out):
    if not Path(out).exists():
        subprocess.run(["gdown", "--fuzzy", f"https://drive.google.com/file/d/{file_id}/view", "-O", out], check=True)
    dest = Path(out).stem
    Path(dest).mkdir(exist_ok=True)
    head = open(out, "rb").read(8)
    if zipfile.is_zipfile(out):
        with zipfile.ZipFile(out) as z:
            z.extractall(dest)
    elif head.startswith(b"Rar!"):          # the Egyptian images come as a .rar file
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True, check=True)
        subprocess.run(["unrar", "x", "-o+", "-idq", out, dest + "/"], check=True)
    else:
        raise SystemExit(f"{out}: unknown file type {head!r} (maybe a Drive quota/HTML page)")

gdrive(CHV_DRIVE_ID, "chv.zip")
gdrive(EGY_DRIVE_ID, "egy.zip")

def split_list(name):
    url = f"{REPO_RAW}/splits/{urllib.parse.quote(name)}"
    return [l.strip() for l in urllib.request.urlopen(url).read().decode().splitlines() if l.strip()]

splits = {f"{d}_{p}": split_list(f"{d}_{p}.txt") for d in ("chv", "egy") for p in ("train", "valid", "test")}
print({k: len(v) for k, v in splits.items()})   # expect chv 1064/133/133, egy 138/17/18

## 3. Build the YOLO folders (CHV labels as-is, Egyptian EXIF rotation baked in)

In [ ]:
import shutil
from PIL import Image, ImageOps

def find_all(root, pattern):
    return {p.name.lower(): p for p in Path(root).rglob(pattern) if "__MACOSX" not in p.parts}

chv_imgs = find_all("chv", "*.jpg")
chv_anns = {p.stem: p for p in Path("chv").rglob("*.txt") if "__MACOSX" not in p.parts and p.stem.startswith("ppe_")}
egy_imgs = {k: v for k, v in find_all("egy", "*").items() if k.endswith((".jpg", ".jpeg", ".png"))}
print("found:", len(chv_imgs), "CHV images,", len(chv_anns), "CHV labels,", len(egy_imgs), "Egyptian images")

D = W / "datasets"
FOLDER = {"train": "train", "valid": "val", "test": "test"}
for part, folder in FOLDER.items():
    (D / "chv/images" / folder).mkdir(parents=True, exist_ok=True)
    (D / "chv/labels" / folder).mkdir(parents=True, exist_ok=True)
    (D / "egy/images" / folder).mkdir(parents=True, exist_ok=True)
    for rel in splits[f"chv_{part}"]:
        name = Path(rel).name
        shutil.copy(chv_imgs[name.lower()], D / "chv/images" / folder / name)
        shutil.copy(chv_anns[Path(name).stem], D / "chv/labels" / folder / (Path(name).stem + ".txt"))
    missing, rotated = [], 0
    for rel in splits[f"egy_{part}"]:
        name = Path(rel).name
        src = egy_imgs.get(name.lower())
        if src is None:
            missing.append(name); continue
        dst = D / "egy/images" / folder / (Path(name).stem + ".jpg")
        with Image.open(src) as im:
            if im.getexif().get(274, 1) != 1:
                rotated += 1
            ImageOps.exif_transpose(im).convert("RGB").save(dst, quality=95)
    print(f"{folder}: egy missing={missing} rotated={rotated}")

Path("chv.yaml").write_text(
    f"path: {D/'chv'}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)))
print(open("chv.yaml").read())

## 4. Train the labelling model on CHV

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data="chv.yaml", epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
            deterministic=True, project=str(W / "runs"), name="labeller", exist_ok=True)
BEST = Path(model.trainer.best)
RUN_DIR = Path(model.trainer.save_dir)
print("best weights:", BEST)

## 5. Check the labelling model on the CHV test set

In [ ]:
best = YOLO(str(BEST))
m = best.val(data="chv.yaml", split="test", imgsz=IMGSZ, project=str(W / "runs"), name="labeller_test", exist_ok=True)
print("CHV test  mAP50 = %.3f   mAP50-95 = %.3f" % (m.box.map50, m.box.map))
for i, n in enumerate(NAMES):
    print(f"  {n:14s} AP50 = {m.box.ap50[i]:.3f}")

## 6. Pre-label the 173 Egyptian images

In [ ]:
import csv
OUT = W / "egy_prelabels"
if OUT.exists(): shutil.rmtree(OUT)
rows = []
for folder in FOLDER.values():
    (OUT / "labels" / folder).mkdir(parents=True)
    imgs = sorted((D / "egy/images" / folder).glob("*.jpg"))
    for r in best.predict([str(p) for p in imgs], imgsz=IMGSZ, conf=CONF, verbose=False, stream=True):
        p = Path(r.path)
        lines = []
        for c, xywhn, conf in zip(r.boxes.cls.tolist(), r.boxes.xywhn.tolist(), r.boxes.conf.tolist()):
            lines.append("%d %.6f %.6f %.6f %.6f" % (int(c), *xywhn))
            rows.append([folder, p.name, int(c), NAMES[int(c)], round(conf, 3), *[round(v, 6) for v in xywhn]])
        (OUT / "labels" / folder / (p.stem + ".txt")).write_text("\n".join(lines) + ("\n" if lines else ""))

with open(OUT / "boxes_with_confidence.csv", "w", newline="") as f:
    csv.writer(f).writerows([["split", "image", "class_id", "class", "conf", "x", "y", "w", "h"], *rows])

# CVAT "YOLO 1.1" import format (all 173 images in one task)
cv = OUT / "cvat_yolo1.1"; (cv / "obj_train_data").mkdir(parents=True)
(cv / "obj.names").write_text("\n".join(NAMES) + "\n")
(cv / "obj.data").write_text(f"classes = {len(NAMES)}\nnames = data/obj.names\ntrain = data/train.txt\n")
listing = []
for t in sorted((OUT / "labels").rglob("*.txt")):
    shutil.copy(t, cv / "obj_train_data" / t.name)
    listing.append(f"data/obj_train_data/{t.stem}.jpg")
(cv / "train.txt").write_text("\n".join(listing) + "\n")

from collections import Counter
cnt = Counter(r[3] for r in rows)
print("label files:", len(list((OUT / 'labels').rglob('*.txt'))), " boxes:", len(rows))
print(dict(cnt))

## 7. Download the result

In [ ]:
shutil.copy(RUN_DIR / "results.csv", OUT / "labeller_training_results.csv")
shutil.make_archive("/content/egy_prelabels", "zip", OUT)
from google.colab import files
files.download("/content/egy_prelabels.zip")